In [1]:
import torch
import torch.nn as nn
from torch.autograd import Variable
from torchvision import datasets, transforms, utils
from torch.utils.data.sampler import SubsetRandomSampler
from torch.nn import functional as F
import matplotlib.pyplot as plt

In [2]:
batch_size = 64
input_size = 784  
num_classes = 1   
num_epochs = 10  
learning_rate = 0.0001 
momentum = 0.0 

In [3]:
train_data = datasets.MNIST('c://pytorch//data/', train=True, download=True,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081,))
                   ]))
test_data = datasets.MNIST('c://pytorch//data/', train=False, download=True,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081,))
                   ]))


In [4]:
subset_indices = ((train_data.train_labels == 0) + (train_data.train_labels == 1)).nonzero().view(-1)

train_loader = torch.utils.data.DataLoader(dataset=train_data, 
                                           batch_size=batch_size, 
                                           shuffle=False,
                                           sampler=SubsetRandomSampler(subset_indices))

subset_indices = ((test_data.test_labels == 0) + (test_data.test_labels == 1)).nonzero().view(-1)

test_loader = torch.utils.data.DataLoader(dataset=test_data, 
                                          batch_size=batch_size,
                                          shuffle=False,
                                          sampler=SubsetRandomSampler(subset_indices))



C:\Users\sanja\anaconda3\Lib\site-packages\torchvision\datasets\mnist.py:66: UserWarning: train_labels has been renamed targets
  warnings.warn("train_labels has been renamed targets")
C:\Users\sanja\anaconda3\Lib\site-packages\torchvision\datasets\mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


In [5]:
class SVM_Loss(nn.modules.Module):    
    def __init__(self):
        super(SVM_Loss,self).__init__()
    def forward(self, outputs, labels):
         return torch.sum(torch.clamp(1 - outputs.t()*labels, min=0))/batch_size

#SVM regression model and Loss
svm_model = nn.Linear(input_size,num_classes)
#model = LogisticRegression(input_size,num_classes)

## Loss criteria and SGD optimizer
svm_loss_criteria = SVM_Loss()

#loss_criteria = nn.CrossEntropyLoss()  

svm_optimizer = torch.optim.SGD(svm_model.parameters(), lr=learning_rate, momentum=momentum)

total_step = len(train_loader)
for epoch in range(num_epochs):
    avg_loss_epoch = 0
    batch_loss = 0
    total_batches = 0
    for i, (images, labels) in enumerate(train_loader):
        # Reshape images to (batch_size, input_size)
        images = images.reshape(-1, 28*28)                      
        labels = Variable(2*(labels.float()-0.5))
                
        # Forward pass        
        outputs = svm_model(images)           
        loss_svm = svm_loss_criteria(outputs, labels)    
        
       
        # Backward and optimize
        svm_optimizer.zero_grad()
        loss_svm.backward()
        svm_optimizer.step()    
        
        #print("Model's parameter after the update:")
        #for param2 in svm_model.parameters():
         #   print(param2)
        total_batches += 1     
        batch_loss += loss_svm.item()

    avg_loss_epoch = batch_loss/total_batches
    print ('Epoch [{}/{}], Averge Loss:for epoch[{}, {:.4f}]' 
                   .format(epoch+1, num_epochs, epoch+1, avg_loss_epoch ))
        


Epoch [1/10], Averge Loss:for epoch[1, 0.2481]
Epoch [2/10], Averge Loss:for epoch[2, 0.0507]
Epoch [3/10], Averge Loss:for epoch[3, 0.0344]
Epoch [4/10], Averge Loss:for epoch[4, 0.0275]
Epoch [5/10], Averge Loss:for epoch[5, 0.0236]
Epoch [6/10], Averge Loss:for epoch[6, 0.0210]
Epoch [7/10], Averge Loss:for epoch[7, 0.0191]
Epoch [8/10], Averge Loss:for epoch[8, 0.0177]
Epoch [9/10], Averge Loss:for epoch[9, 0.0165]
Epoch [10/10], Averge Loss:for epoch[10, 0.0155]


In [6]:
correct = 0.
total = 0.
for images, labels in test_loader:
    images = images.reshape(-1, 28*28)
    
    outputs = svm_model(images)    
    predicted = outputs.data >= 0
    total += labels.size(0) 
    correct += (predicted.view(-1).long() == labels).sum()    
 
print('Accuracy of the model on the test images: %f %%' % (100 * (correct.float() / total)))
print("the learning rate is ", learning_rate)
print( "the momentum is", momentum)

Accuracy of the model on the test images: 99.716309 %
the learning rate is  0.0001
the momentum is 0.0
